In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import math
from torch.utils.data import Dataset, DataLoader

# 1. Load the dataset
text_data = """machine learning models learn patterns from data.
sequence models process data step by step.
recurrent neural networks are designed for sequential tasks.
rnn models maintain hidden states across time steps.
long short term memory networks solve long dependency problems.
lstm uses gates to control information flow.
gru models simplify the lstm architecture.
sequence prediction is useful in many applications.
language modeling predicts the next word in a sentence.
speech recognition processes audio sequences.
time series forecasting predicts future values.
music generation creates new melodies.
generative models learn probability distributions.
they generate new samples similar to training data.
sequence generation is widely used in artificial intelligence.
deep learning improves sequence modeling performance.
transformers revolutionized natural language processing.
the attention mechanism allows models to focus on relevant words.
self attention computes representations of a sequence efficiently.
encoder decoder architectures translate text between languages.
large language models contain billions of parameters.
pretraining on vast amounts of text builds general knowledge.
fine tuning adapts a general model to specific tasks.
word embeddings map vocabulary into continuous vector spaces.
tokenization breaks raw text into manageable pieces.
positional encoding provides order information to transformers.
gradient descent optimizes the neural network weights.
backpropagation computes gradients through time for recurrent networks.
vanishing gradients make training deep networks difficult.
overfitting occurs when a model memorizes the training data.
dropout layers help prevent neural networks from overfitting.
evaluation metrics like perplexity measure language model quality.
cross entropy loss compares predicted probabilities with true labels.
beam search improves sequence decoding over greedy search.
generative adversarial networks synthesize highly realistic data.
variational autoencoders map data into latent continuous spaces.
autoregressive models predict the next token given previous tokens.
natural language understanding extracts meaning from raw text.
sentiment analysis classifies the emotion in a given sentence.
machine translation converts text from one language to another.
chatbots engage in conversational dialogue with human users.
context window size limits how far back a model can look.
batch processing speeds up training on modern graphics cards.
hyperparameter tuning is essential for optimal model performance.
activation functions introduce non linearity into neural networks.
softmax transforms raw scores into a probability distribution."""

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [8]:
# 2. Tokenization & Numerical Representation
# Lowercase and pad punctuation so they are treated as separate tokens
processed_text = text_data.lower().replace('.', ' .')
words = processed_text.split()

# Create vocabulary
vocab = list(set(words))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}
vocab_size = len(vocab)

print(f"Vocabulary Size: {vocab_size}")

# 3. Create input-output sequence pairs
sequence_length = 3  # Use 3 words to predict the next 1 word
X = []
y = []

for i in range(len(words) - sequence_length):
    seq_in = words[i:i + sequence_length]
    seq_out = words[i + sequence_length]
    X.append([word2idx[w] for w in seq_in])
    y.append(word2idx[seq_out])

# Convert to PyTorch tensors
X_tensor = torch.tensor(X, dtype=torch.long)
y_tensor = torch.tensor(y, dtype=torch.long)

# Create DataLoader
class SequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = SequenceDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

Vocabulary Size: 237


In [9]:
# 4. Design LSTM Model
class LSTMGenerator(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(LSTMGenerator, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        # We only care about the output from the last time step to predict the next word
        last_time_step = lstm_out[:, -1, :]
        out = self.fc(last_time_step)
        return out

# Initialize model, loss, and optimizer
lstm_model = LSTMGenerator(vocab_size, embed_size=32, hidden_size=64).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=0.01)

# 5. Train the LSTM model
epochs = 100
print("Training LSTM Model...")
for epoch in range(epochs):
    epoch_loss = 0
    for batch_X, batch_y in dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        outputs = lstm_model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss/len(dataloader):.4f}")

Training LSTM Model...
Epoch [20/100], Loss: 0.0245
Epoch [40/100], Loss: 0.0149
Epoch [60/100], Loss: 0.0139
Epoch [80/100], Loss: 0.0124
Epoch [100/100], Loss: 0.0096


In [10]:
# 6. Generate new sequences
def generate_sequence(model, seed_text, num_words=10):
    model.eval()
    words_gen = seed_text.lower().split()

    with torch.no_grad():
        for _ in range(num_words):
            # Take the last 'sequence_length' words for context
            context = words_gen[-sequence_length:]
            # If seed is too short, pad it (not needed if seed >= sequence_length)
            input_seq = [word2idx.get(w, 0) for w in context]
            input_tensor = torch.tensor([input_seq], dtype=torch.long).to(device)

            output = model(input_tensor)
            predicted_idx = torch.argmax(output, dim=1).item()
            predicted_word = idx2word[predicted_idx]

            words_gen.append(predicted_word)

    return ' '.join(words_gen)

print("\n--- LSTM Generated Sequences ---")
seed = "machine learning models"
print(f"Seed: '{seed}'")
print(f"Generated: {generate_sequence(lstm_model, seed, num_words=8)}")

seed2 = "recurrent neural networks"
print(f"\nSeed: '{seed2}'")
print(f"Generated: {generate_sequence(lstm_model, seed2, num_words=8)}")


--- LSTM Generated Sequences ---
Seed: 'machine learning models'
Generated: machine learning models learn patterns from data . sequence models process

Seed: 'recurrent neural networks'
Generated: recurrent neural networks are designed for sequential tasks . rnn models


In [11]:
# 3. Implement Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # Shape: (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return x

# 4. Design Transformer Architecture
class TransformerGenerator(nn.Module):
    def __init__(self, vocab_size, embed_size, num_heads, hidden_dim, num_layers):
        super(TransformerGenerator, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.pos_encoder = PositionalEncoding(embed_size)

        # Transformer Encoder Layer
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=embed_size,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)

        self.fc_out = nn.Linear(embed_size, vocab_size)

    def forward(self, x):
        # Create a causal mask so the model doesn't look ahead (standard for generation)
        seq_len = x.size(1)
        mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(device)

        x = self.embedding(x) * math.sqrt(x.size(-1))
        x = self.pos_encoder(x)

        # Pass through transformer
        output = self.transformer_encoder(x, mask=mask)

        # Extract the output of the last time step to predict the next word
        last_time_step = output[:, -1, :]
        logits = self.fc_out(last_time_step)
        return logits

# Initialize Transformer
transformer_model = TransformerGenerator(
    vocab_size=vocab_size,
    embed_size=32,
    num_heads=4,
    hidden_dim=64,
    num_layers=2
).to(device)

optimizer_tf = optim.Adam(transformer_model.parameters(), lr=0.005)

In [12]:
# 5. Train the Transformer model
print("Training Transformer Model...")
epochs_tf = 100
for epoch in range(epochs_tf):
    epoch_loss = 0
    for batch_X, batch_y in dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer_tf.zero_grad()
        outputs = transformer_model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer_tf.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs_tf}], Loss: {epoch_loss/len(dataloader):.4f}")

# 6. Generate Sequences using Transformer
print("\n--- Transformer Generated Sequences ---")
seed_tf = "machine learning models"
print(f"Seed: '{seed_tf}'")
print(f"Generated: {generate_sequence(transformer_model, seed_tf, num_words=8)}")

seed_tf2 = "time series forecasting"
print(f"\nSeed: '{seed_tf2}'")
print(f"Generated: {generate_sequence(transformer_model, seed_tf2, num_words=8)}")

Training Transformer Model...
Epoch [20/100], Loss: 0.1231
Epoch [40/100], Loss: 0.2051
Epoch [60/100], Loss: 0.0328
Epoch [80/100], Loss: 0.0579
Epoch [100/100], Loss: 0.0218

--- Transformer Generated Sequences ---
Seed: 'machine learning models'
Generated: machine learning models learn patterns from data . sequence generation is

Seed: 'time series forecasting'
Generated: time series forecasting predicts future values . music generation creates new
